In [1]:
import pandas as pd
import numpy as np
from scipy import signal as sig
from scipy.stats import kurtosis, skew
import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
source_df = pd.read_csv("/kaggle/input/datasets/orvile/satellite-telemetry-data-anomaly-prediction/segments.csv", parse_dates=['timestamp'])
print(f"Number of input rows: {len(source_df)}, and segments: {len(source_df.segment.unique())}")

Number of input rows: 303493, and segments: 2123


In [3]:
def number_of_peaks_finding(array):
    prominence = 0.1 * (np.max(array)-np.min(array))
    peaks = sig.find_peaks(array, prominence=prominence)[0]
    return len(peaks)


def duration(df):
    t1 = pd.Timestamp(df.head(1).timestamp.values[0])
    t2 = pd.Timestamp(df.tail(1).timestamp.values[0])
    return (t2 - t1).seconds


def smooth10_n_peaks(array):
    kernel = np.ones(10)/10
    array_convolved = np.convolve(array, kernel, mode="same")
    return number_of_peaks_finding(array_convolved)


def smooth20_n_peaks(array):
    kernel = np.ones(20)/20
    array_convolved = np.convolve(array, kernel, mode="same")
    return number_of_peaks_finding(array_convolved)


def diff_peaks(array):
    array_diff = np.diff(array)
    return number_of_peaks_finding(array_diff)


def diff2_peaks(array):
    array_diff = np.diff(array, n=2)
    return number_of_peaks_finding(array_diff)


def diff_var(array):
    array_diff = np.diff(array)
    return np.var(array_diff)


def diff2_var(array):
    array_diff = np.diff(array, n=2)
    return np.var(array_diff)


def gaps_squared(df):
    df = df.copy()
    # df["timestamp"] = pd.to_datetime(df["timestamp"])
    df['timestamp2'] = df['timestamp'].shift(1)
    df = df.reset_index().iloc[1:, :]
    df['time_delta'] = (df.timestamp - df.timestamp2).dt.seconds
    df['time_delta_squared'] = df['time_delta']**2
    return df.time_delta_squared.sum()

In [4]:
transformations = {
    "len" : len,
    "mean" : np.mean,
    "var" : np.var,
    "std" : np.std,
    "kurtosis" : kurtosis,
    "skew" : skew,
    "n_peaks" : number_of_peaks_finding,
    "smooth10_n_peaks": smooth10_n_peaks,
    "smooth20_n_peaks": smooth20_n_peaks,
    "diff_peaks" : diff_peaks,
    "diff2_peaks" : diff2_peaks,
    "diff_var" : diff_var,
    "diff2_var" : diff2_var,
}

In [5]:
def generate_dataset(source_df, target_name):
    dataset = []
    for i in tqdm(source_df.segment.unique()):
        res = []
        tdf = source_df.loc[source_df.segment == i, :]
        if tdf.loc[:, "anomaly"].head(1).values == 1:
            anomaly = 1
        else:
            anomaly = 0

        res.append(i)
        res.append(anomaly)
        res.append(tdf.loc[:, "train"].head(1).values[0])
        res.append(tdf.loc[:, "channel"].head(1).values[0])
        res.append(tdf.loc[:, "sampling"].head(1).values[0])
        res.append(duration(tdf))

        for transformation in transformations.values():
            res.append(transformation(tdf.value.values))
        res.append(gaps_squared(tdf))    
        
        dataset.append(res)

    dataset = pd.DataFrame(data=dataset, columns=\
        ["segment", "anomaly", "train", "channel", "sampling", "duration"]
        +list(transformations)+["gaps_squared"])

    dataset["len_weighted"] = dataset["sampling"] * dataset["len"]
    dataset["var_div_duration"] = dataset["var"] / dataset["duration"]
    dataset["var_div_len"] = dataset["var"] / dataset["len"]
    
    dataset.to_csv("/kaggle/working/"+target_name+".csv", index=None)
    return dataset

In [6]:
def oversample_channels(source_df, target_channels, target_segments_per_channel=500, rows_per_segment=50, random_seed=42):
    """
    Oversample specific channels to increase their number of segments.

    Parameters:
    - source_df: original dataframe
    - target_channels: list of channel names to oversample
    - target_segments_per_channel: desired number of segments for each target channel
    - rows_per_segment: number of rows per new segment
    - random_seed: for reproducibility

    Returns:
    - expanded_df: dataframe with new oversampled segments
    """
    np.random.seed(random_seed)
    expanded_segments = []

    current_max_segment = source_df.segment.max() + 1

    for channel in target_channels:
        channel_data = source_df[source_df.channel == channel]
        existing_segments = channel_data.segment.unique()
        new_segment_id = current_max_segment

        while new_segment_id < current_max_segment + target_segments_per_channel:
            # Randomly pick an existing segment of this channel
            seg_id = np.random.choice(existing_segments)
            segment_data = channel_data[channel_data.segment == seg_id].copy()

            # Sample rows for new segment
            if len(segment_data) < rows_per_segment:
                sample = segment_data.sample(n=rows_per_segment, replace=True)
            else:
                sample = segment_data.sample(n=rows_per_segment, replace=False)

            sample['segment'] = new_segment_id
            expanded_segments.append(sample)
            new_segment_id += 1

        current_max_segment = new_segment_id  # update for next channel

    expanded_df = pd.concat([source_df] + expanded_segments, ignore_index=True)
    print(f"Before oversampling: {source_df.groupby('channel')['segment'].nunique()}")
    print(f"After oversampling: {expanded_df.groupby('channel')['segment'].nunique()}")
    
    return expanded_df

In [7]:
# Only oversample CADC0886 and CADC0890
source_df_expanded = oversample_channels(
    source_df,
    target_channels=['CADC0886', 'CADC0890'],
    target_segments_per_channel=500,  # choose how many segments you want
    rows_per_segment=50
)

# Then run your dataset generation
dataset = generate_dataset(source_df_expanded, "__dataset_expanded_channels")
dataset.groupby(by=['train', 'anomaly', 'channel'])['segment'].count()

Before oversampling: channel
CADC0872    546
CADC0873    593
CADC0874    194
CADC0884    158
CADC0886     11
CADC0888    252
CADC0890     14
CADC0892    211
CADC0894    144
Name: segment, dtype: int64
After oversampling: channel
CADC0872    546
CADC0873    593
CADC0874    194
CADC0884    158
CADC0886    511
CADC0888    252
CADC0890    514
CADC0892    211
CADC0894    144
Name: segment, dtype: int64


100%|██████████| 3123/3123 [00:20<00:00, 153.56it/s]


train  anomaly  channel 
0      0        CADC0872    100
                CADC0873    122
                CADC0874     29
                CADC0884     36
                CADC0886    130
                CADC0888     52
                CADC0892     46
                CADC0894     28
       1        CADC0872     32
                CADC0873     31
                CADC0874     23
                CADC0886     50
                CADC0888     12
                CADC0890     80
                CADC0892      7
                CADC0894      5
1      0        CADC0872    315
                CADC0873    366
                CADC0874     96
                CADC0884    122
                CADC0886    243
                CADC0888    140
                CADC0890    116
                CADC0892    131
                CADC0894     95
       1        CADC0872     99
                CADC0873     74
                CADC0874     46
                CADC0886     88
                CADC0888     48
               

In [8]:
dataset.groupby(by=['train', 'channel'])['segment'].count()

train  channel 
0      CADC0872    132
       CADC0873    153
       CADC0874     52
       CADC0884     36
       CADC0886    180
       CADC0888     64
       CADC0890     80
       CADC0892     53
       CADC0894     33
1      CADC0872    414
       CADC0873    440
       CADC0874    142
       CADC0884    122
       CADC0886    331
       CADC0888    188
       CADC0890    434
       CADC0892    158
       CADC0894    111
Name: segment, dtype: int64

In [9]:
import pandas as pd
import os

# ====== CONFIG ======
input_csv = "/kaggle/working/__dataset_expanded_channels.csv"   # your input file
output_dir = "/kaggle/working/channel_batches"  # folder to save files
# ====================

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Load CSV
df = pd.read_csv(input_csv)

# Check if 'anomaly' column exists
if 'anomaly' not in df.columns:
    raise ValueError("Column 'anomaly' not found in CSV")

# Get unique anomaly values
unique_anomalies = df['channel'].unique()

print(f"Found {len(unique_anomalies)} unique anomaly values:")
print(unique_anomalies)

# Split and save
for value in unique_anomalies:
    batch = df[df['channel'] == value]
    
    # Clean filename (in case of weird values)
    safe_value = str(value).replace(" ", "_").replace("/", "_")
    
    output_file = os.path.join(output_dir, f"anomaly_{safe_value}.csv")
    batch.to_csv(output_file, index=False)
    
    print(f"Saved {len(batch)} rows → {output_file}")

print("Done!")

Found 9 unique anomaly values:
['CADC0872' 'CADC0892' 'CADC0874' 'CADC0884' 'CADC0873' 'CADC0886'
 'CADC0888' 'CADC0894' 'CADC0890']
Saved 546 rows → /kaggle/working/channel_batches/anomaly_CADC0872.csv
Saved 211 rows → /kaggle/working/channel_batches/anomaly_CADC0892.csv
Saved 194 rows → /kaggle/working/channel_batches/anomaly_CADC0874.csv
Saved 158 rows → /kaggle/working/channel_batches/anomaly_CADC0884.csv
Saved 593 rows → /kaggle/working/channel_batches/anomaly_CADC0873.csv
Saved 511 rows → /kaggle/working/channel_batches/anomaly_CADC0886.csv
Saved 252 rows → /kaggle/working/channel_batches/anomaly_CADC0888.csv
Saved 144 rows → /kaggle/working/channel_batches/anomaly_CADC0894.csv
Saved 514 rows → /kaggle/working/channel_batches/anomaly_CADC0890.csv
Done!
